In [1]:
import pandas as pd

import matplotlib.pyplot as plot
import matplotlib.dates as mdates
import seaborn as sns
import time
from datetime import datetime, timedelta, date
import teradata
import businessdays as bzd
from pandasql import sqldf

import warnings
warnings.simplefilter('ignore')


start = time.time() 
system = "TERADBAWS.sicredi.net"
user = "P_USR_DL_GESTAO_RECURSOS"
passw = "JAW0b71329aadade35365afc6X"
udaExec = teradata.UdaExec (appName="TESTE", version="1.0", logConsole=False)
session = udaExec.connect(method="odbc", system=system,
        username=user, password=passw, Authentication="TDNEGO");



select_feriados = "Select DT from dl_Gestao_recursos.bas_cad_fer where tipo in ('3','2')"
feriados = pd.read_sql(select_feriados, session)
cal = bzd.Calendar(holidays=feriados.DT,weekdays=['Sunday','Saturday'])

data_inicial = cal.offset(datetime.now(),-21 )###DATA PADRÃO D-60
data_inicial = data_inicial.strftime('%Y-%m-%d') ###DATA PADRÃO D-90

data_final = cal.offset(datetime.now(),-21  )###DATA PADRÃO D-1
data_final = data_final.strftime('%Y-%m-%d') ###DATA PADRÃO D-1

fundos= ['FIRF ALOCACAO','FM MACRO ALOC']
fundo_lista = "','".join(map(str,fundos))

In [2]:

Select = f"""SELECT 
VAR.*
FROM DL_GESTAO_RECURSOS.VAR_ESTRATEGIA_SAS VAR
left join dl_Gestao_recursos.sac_cl_patrsint pl on pl.dt = VAR.DTPOS AND VAR.NOMECARTEIRA = PL.CLCLI_CD
WHERE DT between '{data_inicial}' and '{data_final}' AND VAR.NOMECARTEIRA in ('{fundo_lista}')
AND CODIDENTIFICA IS NOT NULL
 """


df_var = pd.read_sql(Select, session) 
df_var.DtPos = pd.to_datetime(df_var.DtPos, format='%Y-%m-%d') 

pd.set_option('mode.chained_assignment', None)

In [3]:
import pygwalker as pyg
walker = pyg.walk(df_var)

ZeroDivisionError: division by zero

In [ ]:
from pivottablejs import pivot_ui
pivot_ui(df_var) 